# Cuaderno 01: Preprocesamiento y Limpieza de datos (Capa Gold)

**Objetivo General del Proyecto:** Desarrollo de un *pipeline* integral de Machine Learning aplicado a datos de salud reales (UCI) para resolver tres problemas clínicos y operativos distintos.

**Objetivos Específicos (Modelos a Desarrollar)**

*   **1. Clasificación: Predicción de Mortalidad en UCI**
    *   **Propósito:** Modelo estrella en analítica de salud para predecir el riesgo de fallecimiento.
    *   **Variable Objetivo (Target):** `DiedInHospital` (Booleano).
    *   **Variables Predictoras:** Datos demográficos, antropométricos, scores de gravedad clínica (APS, APACHE IV), comorbilidades previas y constantes vitales al momento del ingreso.
*   **2. Regresión: Predicción del Tiempo de Estancia (Length of Stay - LoS)**
    *   **Propósito:** Predecir cuántos días estará un paciente ingresado, lo cual es vital para la gestión predictiva de camas y recursos hospitalarios.
    *   **Variable Objetivo (Target):** `DischargeDayNumber` (Numérico continuo).
*   **3. Clustering: Descubrimiento de Fenotipos Clínicos (No Supervisado)**
    *   **Propósito:** Segmentar a los pacientes en grupos con características similares sin depender de una variable objetivo (etiqueta).
    *   **Enfoque:** Agrupar pacientes utilizando sus diagnósticos de admisión, comorbilidades, constantes vitales y edad para descubrir posibles "clústeres de alto riesgo" subyacentes.

---

## 1. Configuración del Entorno y Librerías
Importación de las herramientas necesarias para la manipulación de datos, conexión a la base de datos y tratamiento de valores nulos.

### 1.1. Uso de librerías
Durante esta actividad se han utilizado las siguientes librerías:

In [1]:
import pandas as pd
import numpy as np
import pyodbc
import yaml
import os
import json
from sklearn.impute import SimpleImputer

### 1.2. Importación de los datos

In [2]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO ('local' o 'cloud')
# ==========================================
ENTORNO = 'local' 

# 1. Cargar las credenciales desde el archivo YAML (subiendo un nivel de directorio)
ruta_config = '../data/config.yaml'

try:
    with open(ruta_config, 'r') as file:
        config = yaml.safe_load(file)
        
    if ENTORNO == 'cloud':
        db_config = config['azure_sql']
        print("Iniciando conexión a Azure SQL...")
    else:
        db_config = config['local_sql']
        print("Iniciando conexión a SQL Server Local...")
        
except FileNotFoundError:
    print(f"Error: No se ha encontrado el archivo en {ruta_config}. Revisa la estructura de carpetas.")
    raise

# 2. Construcción de la cadena de conexión dinámica
# Limpiamos las llaves por si vienen ya incluidas en el YAML
driver_name = db_config['driver'].replace('{', '').replace('}', '')

if ENTORNO == 'cloud':
    connection_string = (
        f"DRIVER={{{driver_name}}};"
        f"SERVER={db_config['server']};"
        f"PORT=1433;"
        f"DATABASE={db_config['database']};"
        f"UID={db_config['username']};"
        f"PWD={db_config['password']}"
    )
else:
    # Conexión local (típicamente con Windows Authentication)
    connection_string = (
        f"DRIVER={{{driver_name}}};"
        f"SERVER={db_config['server']};"
        f"DATABASE={db_config['database']};"
        f"Trusted_Connection={db_config.get('trusted_connection', 'yes')};"
    )

# 3. Conexión y extracción de datos
try:
    conn = pyodbc.connect(connection_string)
    
    # Cargamos la vista vw_Dashboard_DataMining
    query = "SELECT * FROM vw_Dashboard_DataMining"
    df_uci = pd.read_sql(query, conn)
    
    print("¡Conexión exitosa!")
    print("Dimensiones del dataframe:", df_uci.shape)

except Exception as e:
    print(f"Error al conectar o extraer los datos en entorno {ENTORNO}:", e)
finally:
    if 'conn' in locals():
        conn.close()

Iniciando conexión a SQL Server Local...


C:\Users\ainho\AppData\Local\Temp\ipykernel_26464\2919115599.py:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_uci = pd.read_sql(query, conn)


¡Conexión exitosa!
Dimensiones del dataframe: (169530, 52)


In [3]:
display(df_uci)

,Gender,Ethnicity,Age,AdmissionHeight,AdmissionWeight,DischargeWeight,UnitType,UnitStayType,DiagnosisAdmission,Service,...,Creatinine,BUN,Hematocrit,WBC,AdmitYear,AdmitDayNumber,AdmitHour,DischargeYear,DischargeDayNumber,DischargeHour
0,Male,Hispanic,54.0,162.6,63.5,0.00,CTICU,admit,Unknown,unknown,...,3.00,30.0,21.6,9.30,2014,0,11:59:00,2014,0,12:23:00
1,Female,Caucasian,90.0,139.7,0.0,0.00,SICU,readmit,"Effusions, pleural",pulmonary,...,NaN,NaN,NaN,NaN,2014,0,16:46:00,2014,1,12:23:00
2,Male,Caucasian,69.0,175.0,93.0,0.00,SICU,admit,"Renal failure, acute",renal,...,5.04,126.0,30.6,16.50,2014,0,00:11:00,2014,0,12:24:00
3,Female,Other/Unknown,72.0,152.4,55.2,0.00,Med-Surg ICU,readmit,"Sepsis, other",infectious diseases,...,NaN,NaN,NaN,NaN,2014,0,15:54:00,2014,0,12:24:00
4,Male,Hispanic,66.0,175.3,95.2,0.00,Cardiac ICU,admit,"Angina, unstable (angina interferes w/quality ...",cardiovascular,...,1.00,17.0,NaN,NaN,2014,0,14:16:00,2014,0,12:24:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169525,Male,Hispanic,60.0,167.6,64.2,0.00,Med-Surg ICU,admit,"Bleeding, upper GI",gastrointestinal,...,1.72,15.0,23.5,22.50,2014,0,02:47:00,2014,0,08:21:00
169526,Female,Caucasian,84.0,162.6,44.8,44.73,CSICU,admit,Rhythm disturbance (conduction defect),cardiovascular,...,0.61,16.0,43.6,7.70,2014,0,18:34:00,2014,1,08:21:00
169527,Female,African American,54.0,157.0,80.0,0.00,Cardiac ICU,admit,Emphysema/bronchitis,pulmonary,...,0.96,14.0,43.3,9.40,2014,0,13:08:00,2014,3,08:21:00
169528,Female,Caucasian,77.0,154.9,73.3,0.00,Med-Surg ICU,admit,CABG with aortic valve replacement,cardiovascular,...,1.20,18.0,29.8,8.19,2014,0,22:36:00,2014,5,08:21:00


### 1.3. Prevención de Fugas de Datos (Data Leakage)
Antes de comenzar la exploración, procedemos a eliminar aquellas variables que no aportan valor predictivo en el momento del ingreso o que contienen información del futuro (fuga de datos).

* **Metadatos y Fechas:** Se excluyen `AdmitYear`, `DischargeYear` y `AdmitDayNumber` para evitar que el algoritmo aprenda sesgos temporales no clínicos.
* **Fuga de datos por variables de alta:** Se elimina `DischargeHour` ya que la granularidad horaria al alta es información a posteriori. De igual forma, se descarta `DischargeWeight` debido a que es una métrica tomada a la salida de la UCI y presenta una tasa inasumible de nulos (falsos ceros en pacientes fallecidos o traslados urgentes).

In [4]:
# Eliminar columnas que no son predictoras
columnas_a_excluir = [
    'AdmitYear', 'AdmitDayNumber',          # Fechas de ingreso
    'DischargeYear', 'DischargeHour',       # Fechas de alta
    'DischargeWeight'                       # Data Leakage y demasiados nulos (ceros)
]

df_uci_filtrado = df_uci.drop(columns=columnas_a_excluir, errors='ignore')

## 2. Análisis Exploratorio de Datos (EDA)
Fase de diagnóstico inicial para comprender la distribución teórica de las variables, identificar desbalances en las clases y detectar posibles anomalías en los registros médicos.

In [5]:
print(f"Dimensiones iniciales de df_uci_filtrado: {df_uci_filtrado.shape}")

Dimensiones iniciales de df_uci_filtrado: (169530, 47)


In [ ]:
# Separar columnas por su tipo de dato
categoricas = df_uci_filtrado.select_dtypes(include=['object', 'bool']).columns
numericas = df_uci_filtrado.select_dtypes(exclude=['object', 'bool']).columns

### 2.1. Auditoría de Variables Categóricas y Booleanas
Evaluación de las frecuencias en datos demográficos, tipos de ingreso y diagnósticos.

In [ ]:
for col in categoricas:
    print(f"\n[Variable]: {col}")
    # value_counts(dropna=False) nos muestra las categorías y también si hay Nulos
    print(df_uci_filtrado[col].value_counts(dropna=False)) 
    print("-" * 40)


[Variable]: Gender
Gender
Male       91617
Female     77809
              69
Unknown       26
Other          9
Name: count, dtype: int64
----------------------------------------

[Variable]: Ethnicity
Ethnicity
Caucasian           129996
African American     18826
Other/Unknown         7975
Hispanic              6573
Asian                 2844
                      2045
Native American       1271
Name: count, dtype: int64
----------------------------------------

[Variable]: UnitType
UnitType
Med-Surg ICU    93939
MICU            14853
CCU-CTICU       14276
Neuro ICU       12980
Cardiac ICU     10749
SICU            10686
CSICU            6382
CTICU            5665
Name: count, dtype: int64
----------------------------------------

[Variable]: UnitStayType
UnitStayType
admit       153159
readmit       9484
transfer      6887
Name: count, dtype: int64
----------------------------------------

[Variable]: DiagnosisAdmission
DiagnosisAdmission
Sepsis, pulmonary                           

### 2.2. Auditoría de Variables Numéricas
Análisis estadístico descriptivo (medias, desviaciones estándar y cuartiles) de los laboratorios, scores de gravedad y constantes vitales.

In [8]:
# 3. Análisis de Variables Numéricas (Mínimos, Máximos, Medias)
# Usamos .describe() que nos da el count, mean, min, max y los cuartiles
resumen_numerico = df_uci_filtrado[numericas].describe().T

# Si usas Jupyter, display() muestra una tabla HTML muy limpia
display(resumen_numerico)

,count,mean,std,min,25%,50%,75%,max
Age,169501.0,62.967044,17.229949,0.00,53.00,65.0,76.00,90.00
AdmissionHeight,169482.0,166.405628,25.793972,0.00,161.00,170.0,177.80,612.60
AdmissionWeight,169466.0,80.993013,30.629517,0.00,64.50,79.0,95.89,953.00
APS,146891.0,43.632435,23.638396,0.00,27.00,38.0,54.00,200.00
ApacheScore,146891.0,55.492821,25.534811,0.00,37.00,51.0,68.00,211.00
HeartRate_Admission,168639.0,100.303062,30.984813,20.00,87.00,104.0,120.00,220.00
MeanBP_Admission,168238.0,86.692804,41.702478,40.00,53.00,66.0,123.00,200.00
Temperature,159057.0,36.425104,0.927332,20.00,36.20,36.5,36.70,42.29
RespiratoryRate_Admission,167886.0,25.323184,14.992205,4.00,11.00,27.0,36.00,60.00
Eyes,165871.0,3.490393,0.934628,1.00,3.00,4.0,4.00,4.00


### 2.3. Exploración de Valores Atípicos (Outliers Fisiológicos)
En los registros médicos electrónicos (EHR) es habitual encontrar errores de medición o fallos informáticos. A continuación, se auditan los valores extremos cruzándolos con los límites fisiológicos compatibles con la vida.

#### 2.3.1. Datos Biométricos: Altura (`AdmissionHeight`) y Peso (`AdmissionWeight`)

In [9]:
# 1. Filtramos los dos grupos
por_debajo = df_uci[df_uci['AdmissionHeight'] < 90]
por_encima = df_uci[df_uci['AdmissionHeight'] > 280]

# 2. Mostramos los conteos totales
print(f"Pacientes con altura < 100 cm: {len(por_debajo)}")
print(f"Pacientes con altura > 280 cm: {len(por_encima)}")
print("-" * 40)

# 3. Vemos qué valores exactos se están registrando en esos extremos
print("\nValores exactos por debajo de 100 cm (Top 10):")
print(por_debajo['AdmissionHeight'].value_counts().head(10))

print("\nValores exactos por encima de 250 cm (Top 10):")
print(por_encima['AdmissionHeight'].value_counts().head(10))

Pacientes con altura < 100 cm: 3334
Pacientes con altura > 280 cm: 30
----------------------------------------

Valores exactos por debajo de 100 cm (Top 10):
AdmissionHeight
0.00     2908
1.60       21
1.70       21
72.00      14
1.67       14
15.20      12
63.00      11
62.00      10
1.80       10
66.00       9
Name: count, dtype: int64

Valores exactos por encima de 250 cm (Top 10):
AdmissionHeight
507.00    3
509.00    3
505.00    3
500.00    2
600.00    1
508.00    1
504.80    1
465.10    1
511.00    1
297.18    1
Name: count, dtype: int64


In [10]:
# 1. Conteo de valores atípicos evidentes
ceros_peso = df_uci_filtrado[df_uci_filtrado['AdmissionWeight'] <= 30]
muy_altos = df_uci_filtrado[df_uci_filtrado['AdmissionWeight'] > 350]

print(f"Pacientes con peso <= 30: {len(ceros_peso)}")
print(f"Pacientes con peso > 350 kg: {len(muy_altos)}")
print("-" * 40)

# 2. Investigar posibles bebés o errores (< 30)
print("\nTop 10 valores por debajo de 30:")
print(df_uci_filtrado[df_uci_filtrado['AdmissionWeight'] < 30]['AdmissionWeight'].value_counts().head(10))

# 3. Investigar posibles libras (lbs) o bariátricos
print("\nTop 10 valores por encima de 350:")
print(df_uci_filtrado[df_uci_filtrado['AdmissionWeight'] > 350]['AdmissionWeight'].value_counts().head(10))

Pacientes con peso <= 30: 5927
Pacientes con peso > 350 kg: 25
----------------------------------------

Top 10 valores por debajo de 30:
AdmissionWeight
0.0     5760
0.5       17
29.5       6
29.0       4
29.4       4
29.9       4
10.0       3
26.0       3
27.2       3
0.4        3
Name: count, dtype: int64

Top 10 valores por encima de 350:
AdmissionWeight
735.0    1
880.6    1
850.0    1
641.7    1
838.0    1
713.0    1
855.0    1
818.0    1
630.9    1
362.8    1
Name: count, dtype: int64


#### 2.3.2. Constantes Vitales: Hipotermia (`Temperature`)

In [11]:
# Filtramos pacientes con temperaturas sospechosamente bajas (< 30 °C)
hipotermias = df_uci_filtrado[df_uci_filtrado['Temperature'] < 30]

print(f"Pacientes con temperatura < 30 °C: {len(hipotermias)}")

if len(hipotermias) > 0:
    print("\nDistribución de las temperaturas bajas (Top 10):")
    print(hipotermias['Temperature'].value_counts().head(10))

Pacientes con temperatura < 30 °C: 185

Distribución de las temperaturas bajas (Top 10):
Temperature
29.60    11
29.00     9
29.39     8
29.80     7
29.50     6
29.10     6
28.60     4
20.00     4
28.80     4
28.19     4
Name: count, dtype: int64


#### 2.3.3. Laboratorios: Glucosa (`Glucose`) y Sodio (`Sodium`)

In [12]:
bajos_glucosa = df_uci_filtrado[df_uci_filtrado['Glucose'] < 10]
altos_glucosa = df_uci_filtrado[df_uci_filtrado['Glucose'] > 2000]

print(f"Pacientes con Glucosa < 10 mg/dL: {len(bajos_glucosa)}")
print(f"Pacientes con Glucosa > 2000 mg/dL: {len(altos_glucosa)}")
print("-" * 40)

print("\nTop 10 valores de Glucosa por debajo de 10:")
print(df_uci_filtrado[df_uci_filtrado['Glucose'] < 10]['Glucose'].value_counts().head(10))

print("\nTop 10 valores de Glucosa por encima de 2000:")
print(df_uci_filtrado[df_uci_filtrado['Glucose'] > 2000]['Glucose'].value_counts().head(10))

Pacientes con Glucosa < 10 mg/dL: 21
Pacientes con Glucosa > 2000 mg/dL: 1
----------------------------------------

Top 10 valores de Glucosa por debajo de 10:
Glucose
9.0    5
6.0    5
3.0    4
1.0    2
8.0    2
4.0    2
5.0    1
Name: count, dtype: int64

Top 10 valores de Glucosa por encima de 2000:
Glucose
2357.0    1
Name: count, dtype: int64


In [13]:
bajos_sodio = df_uci_filtrado[df_uci_filtrado['Sodium'] < 90]
altos_sodio = df_uci_filtrado[df_uci_filtrado['Sodium'] > 190]

print(f"Pacientes con Sodio < 90 mEq/L: {len(bajos_sodio)}")
print(f"Pacientes con Sodio > 190 mEq/L: {len(altos_sodio)}")
print("-" * 40)

print("\nTop 10 valores de Sodio por debajo de 90:")
print(df_uci_filtrado[df_uci_filtrado['Sodium'] < 90]['Sodium'].value_counts().head(10))

print("\nTop 10 valores de Sodio por encima de 190:")
print(df_uci_filtrado[df_uci_filtrado['Sodium'] > 190]['Sodium'].value_counts().head(10))

Pacientes con Sodio < 90 mEq/L: 0
Pacientes con Sodio > 190 mEq/L: 2
----------------------------------------

Top 10 valores de Sodio por debajo de 90:
Series([], Name: count, dtype: int64)

Top 10 valores de Sodio por encima de 190:
Sodium
195.0    1
194.0    1
Name: count, dtype: int64


## 3. Limpieza y filtrado de datos
A partir de los hallazgos del EDA, se procede a estandarizar los datos, unificar categorías minoritarias y corregir unidades de medida.

### 3.1. Estandarización de Variables Categóricas

##### Gender

In [ ]:
print(f"Dimensiones antes de limpiar Gender: {df_uci_filtrado.shape}")

# 1. Quitamos posibles espacios en blanco invisibles
df_uci_filtrado['Gender'] = df_uci_filtrado['Gender'].str.strip()

# 2. Definimos los valores que queremos descartar
valores_invalidos = ['', 'Other', 'Unknown']

# 3. Filtramos el DataFrame para quedarnos SOLO con los que NO están en esa lista
df_uci_filtrado = df_uci_filtrado[~df_uci_filtrado['Gender'].isin(valores_invalidos)]

# 4. Eliminamos también los valores nulos (NaN) reales si los hubiera
df_uci_filtrado = df_uci_filtrado.dropna(subset=['Gender'])

print(f"Dimensiones tras limpiar Gender: {df_uci_filtrado.shape}")
print("\nDistribución final de Gender:")
print(df_uci_filtrado['Gender'].value_counts())

Dimensiones antes de limpiar Gender: (169530, 47)
Dimensiones tras limpiar Gender: (169426, 47)

Distribución final de Gender:
Gender
Male      91617
Female    77809
Name: count, dtype: int64


##### Ethnicity

In [ ]:
# 1. Quitamos posibles espacios en blanco invisibles
df_uci_filtrado['Ethnicity'] = df_uci_filtrado['Ethnicity'].str.strip()

# 2. Agrupamos los valores vacíos dentro de 'Other/Unknown'
df_uci_filtrado['Ethnicity'] = df_uci_filtrado['Ethnicity'].replace(['', None], 'Other/Unknown')

# 3. Comprobamos el resultado
print("Distribución actualizada de Ethnicity:")
print(df_uci_filtrado['Ethnicity'].value_counts(dropna=False))

Distribución actualizada de Ethnicity:
Ethnicity
Caucasian           129986
African American     18821
Other/Unknown         9932
Hispanic              6572
Asian                 2844
Native American       1271
Name: count, dtype: int64


##### DiagnosisAdmission

In [ ]:
# 1. Limpiamos espacios y agrupamos los vacíos inicialmente
df_uci_filtrado['DiagnosisAdmission'] = df_uci_filtrado['DiagnosisAdmission'].str.strip()
df_uci_filtrado['DiagnosisAdmission'] = df_uci_filtrado['DiagnosisAdmission'].replace(['', None], 'Other Diagnosis')

# 2. Calculamos las frecuencias de cada diagnóstico
conteos_diag = df_uci_filtrado['DiagnosisAdmission'].value_counts()

# 3. Definimos el umbral (puedes ajustar este número)
umbral = 500 

# 4. Obtenemos la lista de diagnósticos minoritarios
diagnosticos_minoritarios = conteos_diag[conteos_diag < umbral].index

# 5. Los agrupamos en "Other Diagnosis"
df_uci_filtrado['DiagnosisAdmission'] = df_uci_filtrado['DiagnosisAdmission'].replace(diagnosticos_minoritarios, 'Other Diagnosis')

# 6. Comprobamos el resultado
print(f"Total de categorías tras la reducción: {df_uci_filtrado['DiagnosisAdmission'].nunique()} (originalmente eran 393)")
print("\nDistribución de las principales categorías:")
print(df_uci_filtrado['DiagnosisAdmission'].value_counts().head(10))

Total de categorías tras la reducción: 70 (originalmente eran 393)

Distribución de las principales categorías:
DiagnosisAdmission
Other Diagnosis                                                                                       36062
Sepsis, pulmonary                                                                                      8422
Infarction, acute myocardial (MI)                                                                      6836
CVA, cerebrovascular accident/stroke                                                                   6287
CHF, congestive heart failure                                                                          6017
Sepsis, renal/UTI (including bladder)                                                                  5018
Diabetic ketoacidosis                                                                                  4688
CABG alone, coronary artery bypass grafting                                                            4473
Cardi

##### PastHistory

In [ ]:
# 1. Definimos las categorías que queremos agrupar
categorias_desconocidas = ['Not Obtainable', 'Not Performed', 'None']

# 2. Reemplazamos esas categorías por una etiqueta unificada
df_uci_filtrado['PastHistory'] = df_uci_filtrado['PastHistory'].replace(
    categorias_desconocidas, 'Unknown/Not Performed'
)

# 3. Por seguridad, si hay verdaderos valores nulos (NaN) en Pandas, los metemos en la misma categoría
df_uci_filtrado['PastHistory'] = df_uci_filtrado['PastHistory'].fillna('Unknown/Not Performed')

# Comprobamos cómo queda la distribución final
print("Distribución de PastHistory tras la agrupación:")
print(df_uci_filtrado['PastHistory'].value_counts())

Distribución de PastHistory tras la agrupación:
PastHistory
Performed                149473
No Health Problems        12348
Unknown/Not Performed      7605
Name: count, dtype: int64


##### AdmitHour

In [ ]:
# 1. Extraemos solo la hora (el número del 0 al 23) de la cadena de texto
# Aseguramos que sea string, extraemos los primeros 2 caracteres y lo pasamos a numérico
horas = df_uci_filtrado['AdmitHour'].astype(str).str[:2].astype(float)

# 2. Creamos las condiciones para los turnos hospitalarios
condiciones = [
    (horas >= 8) & (horas < 16),   # De 08:00 a 15:59
    (horas >= 16) & (horas <= 23), # De 16:00 a 23:59
    (horas >= 0) & (horas < 8)     # De 00:00 a 07:59
]

# 3. Definimos las etiquetas de los tramos
etiquetas = ['Mañana', 'Tarde', 'Noche']

# 4. Aplicamos la categorización y sobrescribimos la columna original
df_uci_filtrado['AdmitHour'] = np.select(condiciones, etiquetas, default='Desconocido')

# Comprobamos cómo ha quedado la distribución
print("Distribución de AdmitHour en tramos horarios:")
print(df_uci_filtrado['AdmitHour'].value_counts())

Distribución de AdmitHour en tramos horarios:
AdmitHour
Tarde     72387
Noche     63863
Mañana    33176
Name: count, dtype: int64


### 3.2. Corrección de Unidades Numéricas (Altura)
Se identifican registros introducidos erróneamente en metros o pulgadas y se unifican al estándar del sistema métrico (centímetros).

##### AdmissionHeight

In [ ]:
print(f"Dimensiones antes de limpiar altura: {df_uci_filtrado.shape}")

# 1. CONVERTIR METROS A CENTÍMETROS
# Buscamos valores mayores a 0 y menores o iguales a 3. 
# (Excluimos el 0 de la máscara porque lo borraremos al final)
mask_metros = (df_uci_filtrado['AdmissionHeight'] > 0) & (df_uci_filtrado['AdmissionHeight'] <= 3)
df_uci_filtrado.loc[mask_metros, 'AdmissionHeight'] = df_uci_filtrado.loc[mask_metros, 'AdmissionHeight'] * 100
print(f"Valores corregidos de metros a cm: {mask_metros.sum()}")

# 2. CONVERTIR PULGADAS A CENTÍMETROS
# Buscamos valores entre 30 y 90 en mayores de 18 años
mask_pulgadas = (
    (df_uci_filtrado['AdmissionHeight'] >= 30) & 
    (df_uci_filtrado['AdmissionHeight'] <= 90) & 
    (df_uci_filtrado['Age'] > 18)
)
df_uci_filtrado.loc[mask_pulgadas, 'AdmissionHeight'] = df_uci_filtrado.loc[mask_pulgadas, 'AdmissionHeight'] * 2.54
print(f"Valores corregidos de pulgadas a cm: {mask_pulgadas.sum()}")

print(f"Dimensiones después de limpiar altura: {df_uci_filtrado.shape}")

Dimensiones antes de limpiar altura: (169426, 47)
Valores corregidos de metros a cm: 138
Valores corregidos de pulgadas a cm: 242
Dimensiones después de limpiar altura: (169426, 47)


### 3.3. Eliminación de Outliers Fisiológicos y Errores de Medición
En esta fase aplicamos un filtrado estricto para limpiar el *dataset* de valores biológicamente imposibles o errores de tecleo evidentes (por ejemplo, niveles de glucosa incompatibles con la vida o puntuaciones del modelo APACHE por encima de su máximo teórico).

Para aplicar buenas prácticas de **MLOps** (Operaciones de Machine Learning) y mantener la trazabilidad clínica, se han seguido estos principios:

* **Externalización de parámetros:** Los rangos válidos `[min, max]` para laboratorios, constantes vitales y escalas (Glasgow, APACHE IV) se definen en un archivo externo (`clinical_limits.json`). Esto permite actualizar los criterios médicos sin alterar el código fuente del *pipeline*.
* **Filtrado estricto:** Se eliminan las filas completas de aquellos pacientes que presenten al menos un valor fuera de los límites fisiológicos establecidos.
* **Preservación de nulos:** El filtro actúa de manera quirúrgica; detecta únicamente valores extremos y conserva intactos los valores faltantes (`NaN`), ya que su ausencia representa un patrón clínico válido que se tratará en la posterior fase de imputación.

In [ ]:
# 1. Cargamos el diccionario de límites
ruta_archivo_limites = '../data/clinical_limits.json'

with open(ruta_archivo_limites, 'r') as archivo:
    limites_lab = json.load(archivo)

print(f"Dimensión inicial del dataset: {df_uci_filtrado.shape}")
print("-" * 45)

# 2. Bucle para filtrar las filas
for col, (min_val, max_val) in limites_lab.items():
    if col in df_uci_filtrado.columns:
        # Creamos una máscara para los valores estrictamente fuera de rango
        # (Esto asegura que los valores NaN no se vean afectados y se conserven)
        mascara_outliers = (df_uci_filtrado[col] < min_val) | (df_uci_filtrado[col] > max_val)
        
        num_outliers = mascara_outliers.sum()
        
        if num_outliers > 0:
            print(f"[{col}]: Eliminando {num_outliers} filas (Límites: {min_val} a {max_val})")
            # Sobrescribimos el dataset quedándonos con las filas que NO son outliers (~)
            df_uci_filtrado = df_uci_filtrado[~mascara_outliers]

print("-" * 45)
print(f"Dimensión final tras la limpieza: {df_uci_filtrado.shape}")

Dimensión inicial del dataset: (169426, 47)
---------------------------------------------
[AdmissionHeight]: Eliminando 2951 filas (Límites: 90 a 280)
[AdmissionWeight]: Eliminando 4274 filas (Límites: 30 a 350)
[Temperature]: Eliminando 169 filas (Límites: 30.0 a 45.0)
[Glucose]: Eliminando 21 filas (Límites: 10 a 2000)
[Sodium]: Eliminando 2 filas (Límites: 90 a 190)
---------------------------------------------
Dimensión final tras la limpieza: (162009, 47)


### 3.4. Consolidación de Variables Booleanas
Se transforman todas las variables clínicas de tipo "flag" (comorbilidades, tratamientos, soporte vital). Ante la ausencia de registro, se asume clínicamente que el paciente no presenta dicha condición (imputación `False` -> `0`).

In [ ]:
# 1. Definimos la lista con todas las columnas que son lógicamente booleanas
columnas_booleanas = [
    # Contexto del Ingreso y Documentación
    'ElectiveSurgery', 'HasPhysicalExam', 'Meds',
    
    # Soporte vital y tratamientos
    'Intubated', 'Vent', 'Dialysis', 'ActiveTreatment', 'Thrombolytics', 'IMA',
    'VentApacheDay', 'IntubApacheDay',
    
    # Comorbilidades
    'Diabetes', 'Cirrhosis', 'HepaticFailure', 'MetastaticCancer', 
    'Leukemia', 'Lymphoma', 'Immunosuppression', 'MI',
    
    # Variable objetivo
    'DiedInHospital' 
]

# 2. Bucle para limpiar y transformar todas de una vez
for col in columnas_booleanas:
    # Comprobamos que la columna exista en el DataFrame para evitar errores
    if col in df_uci_filtrado.columns:
        # Rellenamos nulos con False (0) y convertimos a número entero (True=1, False=0)
        df_uci_filtrado[col] = df_uci_filtrado[col].fillna(False).astype(int)

# 3. Comprobamos el resultado viendo los primeros registros de estas columnas
print("Muestra de variables booleanas transformadas:")
display(df_uci_filtrado[columnas_booleanas].head())

# 4. Verificamos que ya no queden valores nulos en este bloque
nulos_booleanas = df_uci_filtrado[columnas_booleanas].isna().sum()
print("\nNulos restantes en columnas booleanas:")
print(nulos_booleanas[nulos_booleanas > 0])

Muestra de variables booleanas transformadas:


C:\Users\ainho\AppData\Local\Temp\ipykernel_26464\1880925761.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_uci_filtrado[col] = df_uci_filtrado[col].fillna(False).astype(int)
C:\Users\ainho\AppData\Local\Temp\ipykernel_26464\1880925761.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_uci_filtrado[col] = df_uci_filtrado[col].fillna(False).astype(int)
C:\Users\ainho\AppData\Local\Temp\ipykernel_26464\1880925761.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version.

,ElectiveSurgery,HasPhysicalExam,Meds,Intubated,Vent,Dialysis,ActiveTreatment,Thrombolytics,IMA,VentApacheDay,IntubApacheDay,Diabetes,Cirrhosis,HepaticFailure,MetastaticCancer,Leukemia,Lymphoma,Immunosuppression,MI,DiedInHospital
0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1
4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0



Nulos restantes en columnas booleanas:
Series([], dtype: int64)


### 3.5. Auditoría de Distribución y Varianza (Variables Binarias)
En el análisis de datos clínicos, es habitual y esperado que las variables binarias (como tratamientos específicos o enfermedades previas) presenten un fuerte desbalance (por ejemplo, 95% de ceros y 5% de unos). A diferencia de otros sectores, en medicina estos eventos minoritarios suelen tener un **gran poder predictivo** sobre el pronóstico del paciente.

El objetivo de esta auditoría no es buscar variables balanceadas, sino identificar aquellas que presentan una **Varianza Cercana a Cero (Near-Zero Variance)**. Calcularemos la proporción exacta de la clase positiva (1) para detectar columnas cuya frecuencia sea tan extremadamente baja (ej. < 0.5%) que no ofrezcan suficientes ejemplos al algoritmo para aprender patrones útiles, actuando únicamente como ruido estadístico.

In [ ]:
# 1. Creamos una lista vacía para guardar los resultados
resultados_dist = []

# 2. Calculamos los porcentajes para cada columna booleana
for col in columnas_booleanas:
    if col in df_uci_filtrado.columns:
        # value_counts(normalize=True) devuelve el porcentaje en decimales
        dist = df_uci_filtrado[col].value_counts(normalize=True) * 100
        
        pct_0 = dist.get(0, 0.0)
        pct_1 = dist.get(1, 0.0)
        
        resultados_dist.append({
            'Variable': col,
            '% Ceros (0)': round(pct_0, 2),
            '% Unos (1)': round(pct_1, 2)
        })

# 3. Convertimos a DataFrame y lo ordenamos por la rareza del '1'
df_dist = pd.DataFrame(resultados_dist).sort_values(by='% Unos (1)', ascending=False)

print("Distribución de variables binarias:")
display(df_dist)

# 4. Alerta opcional para Varianza Cercana a Cero (ej. menos del 0.5% de prevalencia)
casi_cero = df_dist[df_dist['% Unos (1)'] < 0.5]
if not casi_cero.empty:
    print("\nALERTA: Variables con Varianza Cercana a Cero (< 0.5% de positivos):")
    display(casi_cero)

Distribución de variables binarias:


,Variable,% Ceros (0),% Unos (1)
1,HasPhysicalExam,4.11,95.89
6,ActiveTreatment,41.70,58.30
9,VentApacheDay,66.50,33.50
10,IntubApacheDay,73.59,26.41
4,Vent,75.44,24.56
11,Diabetes,76.69,23.31
0,ElectiveSurgery,82.28,17.72
3,Intubated,84.85,15.15
19,DiedInHospital,90.86,9.14
5,Dialysis,96.28,3.72



ALERTA: Variables con Varianza Cercana a Cero (< 0.5% de positivos):


,Variable,% Ceros (0),% Unos (1)
16,Lymphoma,99.53,0.47


##### Eliminación de Variables con Varianza Cercana a Cero (Near-Zero Variance)
Tras la auditoría de distribución de las variables binarias, comprobamos que la mayoría de los eventos clínicos minoritarios mantienen una prevalencia suficiente para el entrenamiento. Sin embargo, el sistema ha levantado una alerta sobre la variable `Lymphoma`, que presenta un 99.53% de ceros y apenas un **0.47% de casos positivos**.

En el contexto del Machine Learning, mantener variables con una varianza tan cercana a cero no aporta ejemplos suficientes para que los algoritmos aprendan patrones robustos. Al contrario, introducen ruido estadístico, aumentan la dimensionalidad innecesariamente y favorecen el sobreajuste (*overfitting*). Por tanto, siguiendo las mejores prácticas de preparación de datos, procedemos a eliminarla del espacio de características.

In [ ]:
print(f"Dimensiones antes de eliminar Lymphoma: {df_uci_filtrado.shape}")

# Lista de variables a eliminar por varianza cercana a cero
columnas_nzv = ['Lymphoma']

# Eliminamos la columna (usamos errors='ignore' por si se ejecuta la celda dos veces)
df_uci_filtrado = df_uci_filtrado.drop(columns=columnas_nzv, errors='ignore')

# Actualizamos también nuestra lista de columnas booleanas para futuros usos
if 'Lymphoma' in columnas_booleanas:
    columnas_booleanas.remove('Lymphoma')

print(f"Dimensiones después de la limpieza: {df_uci_filtrado.shape}")
print("Variable 'Lymphoma' eliminada con éxito.")

Dimensiones antes de eliminar Lymphoma: (162009, 47)
Dimensiones después de la limpieza: (162009, 46)
Variable 'Lymphoma' eliminada con éxito.


## 4. Tratamiento de Valores Nulos
Antes de proceder a la imputación matemática de los datos faltantes, es imprescindible realizar una auditoría visual de los valores `NaN` (nulos). En esta etapa hemos consolidado todos los nulos (tanto los ausentes de origen como los que generamos al eliminar *outliers* fisiológicos).

El objetivo es cuantificar el volumen de datos faltantes por variable. Aquellas características con un porcentaje crítico de nulos (por ejemplo, > 60-70%) podrían ser candidatas a eliminación para evitar introducir sesgos masivos mediante la imputación. Para el resto de variables numéricas, la proporción de nulos nos confirmará la viabilidad de usar técnicas como la mediana.

In [ ]:
# 1. Calculamos el total de nulos y su porcentaje sobre el total de filas
total_nulos = df_uci_filtrado.isna().sum()
porcentaje_nulos = (total_nulos / len(df_uci_filtrado)) * 100

# 2. Creamos un DataFrame con los resultados
df_nulos = pd.DataFrame({
    'Total_NaN': total_nulos,
    'Porcentaje_%': porcentaje_nulos
})

# 3. Filtramos SOLO las columnas que tienen algún nulo y ordenamos de mayor a menor
df_nulos = df_nulos[df_nulos['Total_NaN'] > 0].sort_values(by='Porcentaje_%', ascending=False)

# 4. Redondeamos el porcentaje a 2 decimales para que sea más legible
df_nulos['Porcentaje_%'] = df_nulos['Porcentaje_%'].round(2)

# 5. Mostramos los resultados
print(f"Hay un total de {len(df_nulos)} variables con valores nulos que requieren atención:")
print("-" * 65)
display(df_nulos)

Hay un total de 18 variables con valores nulos que requieren atención:
-----------------------------------------------------------------


,Total_NaN,Porcentaje_%
WBC,39135,24.16
Hematocrit,35795,22.09
BUN,33294,20.55
Creatinine,32768,20.23
Sodium,32152,19.85
APS,20660,12.75
ApacheScore,20660,12.75
Glucose,19475,12.02
Temperature,9372,5.78
Eyes,3062,1.89


### 4.1. Imputación Robusta (Estrategia: Mediana)
Dada la marcada asimetría inherente a las variables biomédicas en entornos de cuidados críticos (donde los pacientes extremadamente graves sesgan la media), se emplea el algoritmo `SimpleImputer` utilizando la **mediana**. Esto garantiza una imputación matemáticamente estable y clínicamente conservadora.

In [ ]:
# 1. Definimos cuáles son las variables numéricas reales (excluyendo booleanas y categóricas)
# Seleccionamos todas las columnas numéricas
todas_numericas = df_uci_filtrado.select_dtypes(include=[np.number]).columns.tolist()

# Filtramos para quitar las booleanas (que ahora son 0/1) y la variable objetivo si no queremos tocarla
numericas = [col for col in todas_numericas if col not in columnas_booleanas and col != 'DischargeDayNumber']

print(f"Se van a imputar {len(numericas)} variables numéricas.")
print(f"Nulos totales antes de la imputación: {df_uci_filtrado[numericas].isna().sum().sum()}")
print("-" * 50)

# 2. Instanciamos el imputador usando la mediana
imputer = SimpleImputer(strategy='median')

# 3. Aplicamos la imputación SOLO a las variables numéricas seleccionadas
df_uci_filtrado[numericas] = imputer.fit_transform(df_uci_filtrado[numericas])

# 4. Comprobación final
nulos_restantes = df_uci_filtrado[numericas].isna().sum().sum()
print(f"Nulos en variables numéricas tras la imputación: {nulos_restantes}")
print(f"Dimensiones del dataset final: {df_uci_filtrado.shape}")

Se van a imputar 18 variables numéricas.
Nulos totales antes de la imputación: 255557
--------------------------------------------------
Nulos en variables numéricas tras la imputación: 0
Dimensiones del dataset final: (162009, 46)


## 5. Exportación del "Gold Dataset" (Punto de Control)
Tras completar exhaustivamente las fases de limpieza, manejo de valores atípicos e imputación de nulos, el *dataset* ha alcanzado su estado "Gold" (listo para el modelado). 

Para garantizar la reproducibilidad y la integridad de la memoria RAM, se exporta esta matriz resultante. Los siguientes *notebooks* de experimentación partirán de este punto de control.

In [26]:
print("Preparando la exportación de los datos...")

# 1. Creamos una carpeta para los datos procesados si no existe en tu repositorio
carpeta_destino = 'datos_procesados'
os.makedirs(carpeta_destino, exist_ok=True)

# 2. Definimos las rutas de los archivos
ruta_csv = f"{carpeta_destino}/uci_gold_dataset.csv"
ruta_parquet = f"{carpeta_destino}/uci_gold_dataset.parquet"

# 3. Exportamos a CSV (con index=False para no generar la molesta columna 'Unnamed: 0')
df_uci_filtrado.to_csv(ruta_csv, index=False)
print(f"Guardado con éxito en formato CSV: {ruta_csv}")

# 4. Exportamos a Parquet (Altamente recomendado para ML: preserva los tipos de datos y comprime el tamaño)
try:
    df_uci_filtrado.to_parquet(ruta_parquet, index=False)
    print(f"Guardado con éxito en formato Parquet: {ruta_parquet}")
except ImportError:
    print("No se pudo guardar en Parquet. Para habilitarlo, ejecuta 'pip install pyarrow' en tu consola.")

print("-" * 50)
print("¡Fase de Data Cleaning finalizada! Ya puedes cerrar este notebook.")

Preparando la exportación de los datos...
Guardado con éxito en formato CSV: datos_procesados/uci_gold_dataset.csv
Guardado con éxito en formato Parquet: datos_procesados/uci_gold_dataset.parquet
--------------------------------------------------
¡Fase de Data Cleaning finalizada! Ya puedes cerrar este notebook.
